# Multi-Subject H-Reflex Across Days

Reads one analysis pickle per subject (written by **Save Analysis to Disk** in `Read_H-Reflex_App.ipynb` / `Read_H-Reflex_App_Simplified.ipynb`) and plots every subject on one figure:

- **thin line per subject**: that subject's mean over all trials of each day
- **thick line**: mean across subjects for each day, with **SEM** bars (n = subjects that have the day)

Each subject keeps its own pipeline (M/H windows, trial exclusion, pre-filter) exactly as saved in its pickle. The x-axis day is the recording label from that subject's `RECORDING_DIRS` (`-5`…`-1` = baseline, `1`… = conditioning). To change a subject's windows or exclusions, re-run its Read notebook and re-save the pickle.

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import ToggleButtons, Checkbox, Output, VBox, HBox
from helpers import (load_subject_caches, multi_subject_daily_table,
                     plot_multi_subject_daily, format_multi_subject_summary)

In [2]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Subject name → its analysis pickle (paths relative to this notebook's folder).
SUBJECT_PICKLES = {
    'HRPilot-27': r'Up-Conditioning_VNS\HRPilot-27\HRPilot-27_h_reflex_analysis.pkl',
    'HRPilot-31': r'Up-Conditioning_VNS\HRPilot-31\HRPilot-31_h_reflex_analysis.pkl',
    'HRPilot-33': r'Up-Conditioning_VNS\HRPilot-33\HRPilot-33_h_reflex_analysis.pkl',
    'HRPilot-36': r'Up-Conditioning_VNS\HRPilot-36\HRPilot-36_h_reflex_analysis.pkl',
    
}
# Stages whose trials count toward a day (Frequency Test trials are left out).
STAGES = ('control_mode', 'up_cond_vns')
# A day's pooled mean/SEM is drawn only when at least this many subjects have that day.
MIN_SUBJECTS = 2
GROUP_COLOR = '#d62728'   # one treatment group → one color (conditioning days >= 1)
                          # baseline days (<= 0) are always gray
GROUP_LABEL = 'CLV'       # bar drawn above days >= 1 (None to hide)
SAVE_FIG_PATH = None      # e.g. 'multi_subject_h_reflex.png' / '.svg' / '.pdf'

In [3]:
# ── Load every subject's comparison cache ─────────────────────────────────────
subjects = load_subject_caches(SUBJECT_PICKLES)

Loaded Up-Conditioning_VNS\HRPilot-27\HRPilot-27_h_reflex_analysis.pkl  (format 1, saved 2026-11-09T18:30:19)
  contents: ['config', 'trials_df', 'xr_df', 'xr_summary', 'settings_df', 'excluded_trials_df', 'xr_cache', 'xr_pool_cache', 'recordings_raw', 'recordings']
  recordings: ['-5', '-4', '-3', '-2', '-1', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14']  (no raw EMG blocks)
  HRPilot-27: 19 recordings · H 6–10 ms · M 2–4 ms
Loaded Up-Conditioning_VNS\HRPilot-31\HRPilot-31_h_reflex_analysis.pkl  (format 1, saved 2026-11-09T18:29:09)
  contents: ['config', 'trials_df', 'xr_df', 'xr_summary', 'settings_df', 'excluded_trials_df', 'xr_cache', 'xr_pool_cache', 'recordings_raw', 'recordings']
  recordings: ['-5', '-4', '-3', '-2', '-1', '1', '2', '3', '4']  (no raw EMG blocks)
  HRPilot-31: 9 recordings · H 6–10 ms · M 2–4 ms
Loaded Up-Conditioning_VNS\HRPilot-33\HRPilot-33_h_reflex_analysis.pkl  (format 1, saved 2026-11-09T18:05:06)
  contents: ['config', 'tria

In [4]:
# ── H-Reflex Size Across Days: all subjects ───────────────────────────────────
# "% of baseline" divides each subject by the mean of its own baseline days (day < 0),
# so subjects with different absolute EMG amplitudes can be averaged together.
# "Compact" shrinks the figure; "Large text" makes all text bold and much larger with
# thicker lines (posters / slides seen from far away). They can be combined.
# The table under the plot is tab-separated — select + copy, pastes into Excel/Sheets.
_ms_metric = ToggleButtons(options=[('H-Reflex Size', 'h_reflex'), ('M-Wave Size', 'm_wave'),
                                    ('Background MRA', 'background')],
                           description='Metric:', style={'button_width': '160px'})
_ms_norm = Checkbox(value=False, description='% of baseline', indent=False)
_ms_compact = Checkbox(value=False, description='Compact', indent=False)
_ms_large = Checkbox(value=False, description='Large text', indent=False)
_ms_out = Output()

def _ms_refresh(*_):
    norm = _ms_norm.value
    per, pooled = multi_subject_daily_table(subjects, _ms_metric.value, STAGES,
                                            normalize=norm, min_subjects=MIN_SUBJECTS)
    with _ms_out:
        _ms_out.clear_output(wait=True)
        fig = plot_multi_subject_daily(per, pooled, _ms_metric.value, normalize=norm,
                                       color=GROUP_COLOR, group_label=GROUP_LABEL,
                                       compact=_ms_compact.value, large_text=_ms_large.value)
        if SAVE_FIG_PATH:
            fig.savefig(SAVE_FIG_PATH, dpi=300, bbox_inches='tight')
            print(f'Saved → {SAVE_FIG_PATH}')
        plt.show()
        print(format_multi_subject_summary(per, pooled, unit='%' if norm else 'µV'))

_ms_metric.observe(_ms_refresh, names='value')
for _w in (_ms_norm, _ms_compact, _ms_large):
    _w.observe(_ms_refresh, names='value')
display(VBox([HBox([_ms_metric, _ms_norm, _ms_compact, _ms_large]), _ms_out]))
_ms_refresh()